# Evaluación del modelo

En este notebook se evalúa el modelo entrenado utilizando el conjunto de validación. Se calculan métricas como precisión, recall, F1-score y matriz de confusión para analizar el rendimiento del modelo en la clasificación de noticias relacionadas con la salud.

### Definir importaciones necesarias

In [ ]:
import os

import matplotlib.pyplot as plt
import seaborn as sns
import torch
from sklearn.metrics import classification_report, confusion_matrix
from transformers import BertForSequenceClassification, BertTokenizer

from ml.utils.load_data import load_dataset
from ml.utils.preprocess import preprocess_data

### Realizar las predicciones del modelo en el conjunto de prueba

In [ ]:
# Obtener la ruta del modelo entrenado
current_dir = os.getcwd()
if current_dir.endswith('notebooks'):
    base_dir = os.path.dirname(os.path.dirname(current_dir))
elif current_dir.endswith('src'):
    base_dir = os.path.dirname(current_dir)
else:
    base_dir = current_dir
model_path = os.path.join(base_dir, "models", "bert_classifier")

# Cargar el modelo entrenado
print(f"Cargando modelo desde: {model_path}")
model = BertForSequenceClassification.from_pretrained(model_path)
tokenizer = BertTokenizer.from_pretrained(model_path)

# Cargar los datos de test
df = preprocess_data(load_dataset("test"))
texts = df['text'].tolist()
labels = df['label'].tolist()

# Realizar las predicciones
inputs = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
with torch.no_grad():
    logits = model(**inputs).logits
preds = torch.argmax(logits, dim=1).numpy()

### Mostrar métricas de evaluación y matriz de confusión

In [ ]:
cm = confusion_matrix(labels, preds)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['True', 'False'], yticklabels=['True', 'False'])
plt.ylabel('Realidad')
plt.xlabel('Predicción del Modelo')
plt.title('Matriz de Confusión - Fake News Detector')
plt.show()

# Imprimir informe detallado con las métricas
print("\n--- INFORME DE RESULTADOS (TEST) ---")
print(classification_report(labels, preds, target_names=['Noticia Real (0)', 'Fake News (1)']))